## 项目方案：复合因子（市值因子简单平均赋分）

In [20]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

capital_dt = pd.read_parquet('data/capital.parquet')
capital_dt=capital_dt.sort_values(by=['stock_code', 'change_date'], ascending=[True, True])
capital_dt.rename(columns={'change_date': 'trade_date'}, inplace=True)
price_dt = pd.read_parquet('data/eod_prices.parquet')
price_dt=price_dt.sort_values(by=['stock_code', 'trade_date'], ascending=[True, True])
mv_factor_dt=pd.merge(price_dt, capital_dt, on=['stock_code', 'trade_date'], how='left')
mv_factor_dt.fillna(method='ffill', inplace=True)
mv_factor_dt.head()

mv_factor_dt['float_a_shares_mv']=mv_factor_dt['float_a_shares']*mv_factor_dt['close_price']
mv_factor_dt['total_a_shares_mv']=mv_factor_dt['total_a_shares']*mv_factor_dt['close_price']
mv_factor_dt.tail()

import numpy as np
mv_factor_dt.dropna(inplace=True)
mv_factor_dt['mv_flt']=-np.log(mv_factor_dt['float_a_shares_mv'])
mv_factor_dt['mv_tot']=-np.log(mv_factor_dt['total_a_shares_mv'])
mv_factor_dt.head()

# 按交易日期分组，对当天全股票池的mv_flt列做Z-score标准化（(值-均值)/标准差）
mv_factor_dt['mv_flt_std'] = mv_factor_dt.groupby('trade_date')['mv_flt'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)

mv_factor_dt['mv_tot_std'] = mv_factor_dt.groupby('trade_date')['mv_tot'].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0  # 避免标准差为0时除以0报错
)
#简单复合
mv_factor_dt['mv_comp']=0.5*mv_factor_dt['mv_flt_std']+0.5*mv_factor_dt['mv_tot_std']
mv_factor_dt.head()



,stock_code,trade_date,close_price,adjusted_close,adj_factor,volume,turnover,total_shares,float_shares,total_a_shares,float_a_shares,float_a_shares_mv,total_a_shares_mv,mv_flt,mv_tot,mv_flt_std,mv_tot_std,mv_comp
102,000001.SZ,20130614,19.06,423.2233,22.204789,328553.36,6.251254e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.918767e+06,9.765651e+06,-15.593639,-16.094382,-1.748887,-2.223469,-1.986178
103,000001.SZ,20130617,19.24,427.2201,22.204789,309210.22,5.961902e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,-15.603038,-16.103781,-1.755373,-2.231283,-1.993328
104,000001.SZ,20130618,19.73,438.1005,22.204789,389599.38,7.701559e+05,512437.850182,310502.606910,512363.631057,310533.405247,6.126824e+06,1.010893e+07,-15.628187,-16.128930,-1.778689,-2.255429,-2.017059
105,000001.SZ,20130619,19.24,427.2201,22.204789,369977.53,7.140700e+05,512437.850182,310502.606910,512363.631057,310533.405247,5.974663e+06,9.857876e+06,-15.603038,-16.103781,-1.761415,-2.235223,-1.998319
106,000001.SZ,20130620,11.18,400.6981,35.840616,903066.54,1.029738e+06,819808.237690,496861.283537,819958.402672,496892.496412,5.555258e+06,9.167135e+06,-15.530255,-16.031135,-1.730716,-2.199407,-1.965061


In [28]:
from scipy.stats.mstats import winsorize

# 读取数据）
fund_df = pd.read_parquet('data/ref_data/ETF_hold_512100.SH.parquet')

# ===================== 1. 数据预处理 =====================
# 处理mv_factor_dt的日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))

# 处理基金持仓数据的日期格式
fund_df = fund_df.rename(columns={'end_date': 'report_end_date'})
fund_df['report_end_date'] = pd.to_datetime(fund_df['report_end_date'])

# 生成基金持仓的有效时间区间（半年报：1-6月，年报：7-12月）
def get_position_valid_period(row):
    year = row['report_year']
    if row['report_type'] == '中报':
        start_date = pd.to_datetime(f'{year}-01-01')
        end_date = pd.to_datetime(f'{year}-06-30')
    elif row['report_type'] == '年报':
        start_date = pd.to_datetime(f'{year}-07-01')
        end_date = pd.to_datetime(f'{year}-12-31')
    else:
        start_date = end_date = row['report_end_date']
    return pd.Series([start_date, end_date], index=['pos_start_date', 'pos_end_date'])

fund_df[['pos_start_date', 'pos_end_date']] = fund_df.apply(get_position_valid_period, axis=1)

# 提取中证1000基金持仓的股票代码+有效区间（去重）
fund_holdings = fund_df[['stock_code', 'pos_start_date', 'pos_end_date']].drop_duplicates()

# ===================== 2. 核心选股逻辑（分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 筛选当日属于中证1000基金持仓的股票
    valid_holdings = fund_holdings[
        (fund_holdings['pos_start_date'] <= trade_date) &
        (fund_holdings['pos_end_date'] >= trade_date)
    ]['stock_code'].tolist()
    holding_stocks = daily_data[daily_data['stock_code'].isin(valid_holdings)]
    
    if len(holding_stocks) == 0:
        return pd.DataFrame()
    
    # 对mv_comp列做1%/99%缩尾
    # holding_stocks['mv_comp'] = winsorize(
    #     holding_stocks['mv_comp'].values, 
    #     limits=(0.01, 0.01),
    #     inclusive=(True, True)
    # )
    
    # 计算分位数并标记多空
    q1 = holding_stocks['mv_flt_std'].quantile(0.20)
    q2 = holding_stocks['mv_flt_std'].quantile(0.80)
    
    holding_stocks['signal'] = 'none'
    holding_stocks.loc[holding_stocks['mv_comp'] <= q1, 'signal'] = 'long'
    holding_stocks.loc[holding_stocks['mv_comp'] >= q2, 'signal'] = 'short'
    
    return holding_stocks

# 按交易日分组执行选股
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 
                                       'mv_comp', 'signal']]
final_selection_df.to_parquet('records/mv_selection_v3.parquet', index=False)


In [29]:
import pandas as pd
import numpy as np
import warnings
from util import *

warnings.filterwarnings("ignore")

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000
START_DATE = '2016-07-01'
END_DATE   = '2025-06-30'
REBALANCE_FREQ = 5

PRICE_FILE = 'data/eod_prices.parquet'
SELECTION_FILE = 'records/mv_selection_v3.parquet'

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)

# Load selection file directly
selection_df = pd.read_parquet(SELECTION_FILE)
selection_df = selection_df[['trade_date', 'stock_code', 'signal']].copy()
selection_df['trade_date'] = pd.to_datetime(selection_df['trade_date'])

# Price pivot
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Rebalance dates
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available.")
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

all_dates = price_pivot.index.tolist()
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# Benchmark
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# ----------------------------
# Helper to get all stocks by signal
# ----------------------------
def get_stocks_by_signal(date, signal_type):
    day_data = selection_df[selection_df['trade_date'] == date]
    day_data = day_data[day_data['signal'] == signal_type]
    return day_data['stock_code'].tolist()

# ----------------------------
# Backtest function (long‑only)
# ----------------------------
def run_long_only_backtest():
    cash = INITIAL_CASH
    holdings = {}
    portfolio_values = []

    for date in timeline:
        today_prices = price_pivot.loc[date]

        if date in rebalance_dates:
            # Close existing positions
            for stock, shares in holdings.items():
                price = today_prices.get(stock, np.nan)
                if pd.notna(price):
                    cash += shares * price
            holdings = {}

            # Build new portfolio
            long_stocks = get_stocks_by_signal(date, 'long')
            valid = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
            if valid:
                amount_per_stock = cash / len(valid)
                for stock in valid:
                    price = today_prices[stock]
                    holdings[stock] = amount_per_stock / price
                cash = 0.0

        # Daily valuation
        portfolio_value = cash
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                portfolio_value += shares * price
        portfolio_values.append(portfolio_value)

    # Build results
    portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')
    benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
    benchmark_funds = INITIAL_CASH * benchmark_series
    portfolio_returns = portfolio_series.pct_change().dropna()
    metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)

    return portfolio_series, benchmark_funds, metrics

# ----------------------------
# Run long‑only backtest
# ----------------------------
print("\n" + "="*50)
print("Running Long‑Only Backtest (all long signals)")
print("="*50)
port_long, bench_long, metrics_long = run_long_only_backtest()

print("\n--- Performance Metrics (Long‑Only) ---")
for k, v in metrics_long.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# ----------------------------
# Plot results with excess returns
# ----------------------------
# Ensure plot_results is defined (or imported from util if available)
# We'll assume the enhanced plot_results is available, either from previous definition
# or from util. If not, you need to define it before calling.
plot_results(port_long, bench_long, timeline, title='Long‑Only (all long signals)', plot_excess=True)

Loading data...
Number of rebalance dates: 437

Running Long‑Only Backtest (all long signals)

--- Performance Metrics (Long‑Only) ---
Annualized Return: -0.0508
Volatility: 0.2364
Sharpe Ratio: -0.2147
Max Drawdown: -0.5393


中证1000的小市值不够小，没有小盘股的收益优势

## 更换股票池：全股票池
应剔除ST、北交所股票

In [30]:
from scipy.stats.mstats import winsorize
import pandas as pd

# ===================== 1. 数据预处理 =====================
# 处理因子数据日期格式（数字转日期）
mv_factor_dt['trade_date'] = pd.to_datetime(mv_factor_dt['trade_date'].astype(str))
mv_factor_dt=mv_factor_dt[mv_factor_dt['stock_code'].str.endswith('SZ')|mv_factor_dt['stock_code'].str.endswith('SH')]

# ===================== 2. 核心选股逻辑（全股票 分位数多空） =====================
def select_stocks_by_date(date_group):
    trade_date = date_group.name
    daily_data = date_group.copy()
    
    # 【无股票池限制】直接使用当日全部股票数据
    if len(daily_data) == 0:
        return pd.DataFrame()
    
    # 对mv_comp列做1%/99%缩尾（可选，取消注释即可启用）
    # daily_data['mv_comp'] = winsorize(
    #     daily_data['mv_comp'].values, 
    #     limits=(0.01, 0.01),
    #     inclusive=(True, True)
    # )
    
    # 计算全股票分位数并标记多空
    q1 = daily_data['mv_flt_std'].quantile(0.01)
    q2 = daily_data['mv_flt_std'].quantile(0.99)
    
    daily_data['signal'] = 'none'
    daily_data.loc[daily_data['mv_comp'] <= q1, 'signal'] = 'long'   # 多头
    daily_data.loc[daily_data['mv_comp'] >= q2, 'signal'] = 'short'  # 空头
    
    return daily_data

# 按交易日分组执行选股（全市场股票）
selected_stocks = mv_factor_dt.groupby('trade_date').apply(select_stocks_by_date)

# 重置索引
selected_stocks = selected_stocks.reset_index(drop=True)

# 保留核心列
final_selection_df = selected_stocks[['trade_date', 'stock_code', 
                                       'mv_comp', 'signal']]
# 保存选股结果
final_selection_df.to_parquet('records/mv_selection_v3.parquet', index=False)

In [47]:
import pandas as pd
import numpy as np
import warnings
from util import *

warnings.filterwarnings("ignore")

# ----------------------------
# User‑configurable parameters
# ----------------------------
INITIAL_CASH = 1_000_000
START_DATE = '2016-07-01'
END_DATE   = '2025-06-30'
REBALANCE_FREQ = 5

PRICE_FILE = 'data/eod_prices.parquet'
SELECTION_FILE = 'records/mv_selection_v3.parquet'

# ----------------------------
# Load and prepare data
# ----------------------------
print("Loading data...")
price_df = load_and_preprocess_price(PRICE_FILE)

# Load selection file directly
selection_df = pd.read_parquet(SELECTION_FILE)
selection_df = selection_df[['trade_date', 'stock_code', 'signal']].copy()
selection_df['trade_date'] = pd.to_datetime(selection_df['trade_date'])

# Price pivot
price_pivot = price_df.pivot(index='trade_date', columns='stock_code', values='adjusted_close')
price_pivot = price_pivot.sort_index()
price_pivot.index = pd.to_datetime(price_pivot.index)

# Filter by date range
price_pivot = price_pivot.loc[START_DATE:END_DATE]
selection_df = selection_df[(selection_df['trade_date'] >= START_DATE) &
                            (selection_df['trade_date'] <= END_DATE)]

# Rebalance dates
selection_dates = sorted(selection_df['trade_date'].unique())
if not selection_dates:
    raise ValueError("No selection data available.")
rebalance_dates = selection_dates[::REBALANCE_FREQ]
print(f"Number of rebalance dates: {len(rebalance_dates)}")

all_dates = price_pivot.index.tolist()
first_rebalance = rebalance_dates[0]
timeline = [d for d in all_dates if d >= first_rebalance]

# Benchmark
benchmark_daily, benchmark_cum = compute_benchmark_returns(price_pivot, START_DATE, END_DATE)

# ----------------------------
# Helper to get all stocks by signal
# ----------------------------
def get_stocks_by_signal(date, signal_type):
    day_data = selection_df[selection_df['trade_date'] == date]
    day_data = day_data[day_data['signal'] == signal_type]
    return day_data['stock_code'].tolist()

# ----------------------------
# Backtest function (long‑only)
# ----------------------------
def run_long_only_backtest():
    cash = INITIAL_CASH
    holdings = {}
    portfolio_values = []

    for date in timeline:
        today_prices = price_pivot.loc[date]

        if date in rebalance_dates:
            # Close existing positions
            for stock, shares in holdings.items():
                price = today_prices.get(stock, np.nan)
                if pd.notna(price):
                    cash += shares * price
            holdings = {}

            # Build new portfolio
            long_stocks = get_stocks_by_signal(date, 'long')
            valid = [s for s in long_stocks if pd.notna(today_prices.get(s, np.nan))]
            if valid:
                amount_per_stock = cash / len(valid)
                for stock in valid:
                    price = today_prices[stock]
                    holdings[stock] = amount_per_stock / price
                cash = 0.0

        # Daily valuation
        portfolio_value = cash
        for stock, shares in holdings.items():
            price = today_prices.get(stock, np.nan)
            if pd.notna(price):
                portfolio_value += shares * price
        portfolio_values.append(portfolio_value)

    # Build results
    portfolio_series = pd.Series(portfolio_values, index=timeline, name='Portfolio')
    benchmark_series = benchmark_cum.reindex(timeline, method='ffill').fillna(1.0)
    benchmark_funds = INITIAL_CASH * benchmark_series
    portfolio_returns = portfolio_series.pct_change().dropna()
    metrics = compute_metrics(portfolio_returns, rf=0, periods_per_year=252)

    return portfolio_series, benchmark_funds, metrics

# ----------------------------
# Run long‑only backtest
# ----------------------------
print("\n" + "="*50)
print("Running Long‑Only Backtest (all long signals)")
print("="*50)
port_long, bench_long, metrics_long = run_long_only_backtest()

print("\n--- Performance Metrics (Long‑Only) ---")
for k, v in metrics_long.items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")

# ----------------------------
# Plot results with excess returns
# ----------------------------
# Ensure plot_results is defined (or imported from util if available)
# We'll assume the enhanced plot_results is available, either from previous definition
# or from util. If not, you need to define it before calling.
plot_results(port_long, bench_long, timeline, title='Long‑Only (all long signals)', plot_excess=True)

Loading data...
Number of rebalance dates: 437

Running Long‑Only Backtest (all long signals)

--- Performance Metrics (Long‑Only) ---
Annualized Return: 0.0777
Volatility: 0.1770
Sharpe Ratio: 0.4388
Max Drawdown: -0.3869
